##### Full Data Profiling
- Combine and read all 4 quarterly CSVs (2023.02-2023.12), then use `SUMMARIZE` to get min/max/unique count/null % per column at a glance

In [2]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

files = [
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202302_to_202304 (1).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202305_to_202307 (2).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202308_to_202310 (1).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202311_to_202312 (1).csv",
]

file_list_sql = ", ".join(f"'{f}'" for f in files)

profile = duckdb.sql(f"""
    SELECT column_name, min, max, approx_unique, null_percentage
    FROM (SUMMARIZE SELECT * EXCLUDE (sha2_hash) FROM read_csv([{file_list_sql}], all_varchar=true, union_by_name=true))
""").df()

profile

RuntimeError: Query interrupted

##### Findings
* `AGMT_END_YMD` (supposed to contain date values) is mixed with non-date values such as `'무약정 (No-contract)'` and `'정보없음 (N/A)'`.
* Count columns such as `TV_SCRB` and `ANALOG_SCRB` contain empty-string values (`""`).




##### Categorical Column Drift Check
- For 23 categorical columns, check whether each value exists across all 4 quarters
- Filter down to only the values that appear in just some quarters (possible definition drift)

In [ ]:
import duckdb
import pandas as pd

files = [
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202302_to_202304 (1).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202305_to_202307 (2).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202308_to_202310 (1).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202311_to_202312 (1).csv",
]


categorical_cols = [
    "SVC_USE_DAYS_GRP", "MEDIA_NM_GRP", "PROD_NM_GRP", "PROD_OLD_YN", "PROD_ONE_PLUS_YN",
    "AGMT_KIND_NM", "STB_RES_1M_YN", "SVOD_SCRB_CNT_GRP", "PAID_CHNL_CNT_GRP", "SCRB_PATH_NM_GRP",
    "INHOME_RATE", "AGMT_END_SEG", "BUNDLE_YN", "DIGITAL_GIGA_YN", "DIGITAL_ALOG_YN",
    "CH_LAST_DAYS_BF_GRP", "VOC_TOTAL_MONTH1_YN", "VOC_STOP_CANCEL_MONTH1_YN", "AGE_GRP10",
    "EMAIL_RECV_CLS_NM", "SMS_SEND_CLS_NM", "NFX_USE_YN", "YTB_USE_YN", "cancel_yn", "CH_FAV_RNK1",
]

file_list_sql = ", ".join(f"'{f}'" for f in files)
unpivot_cols_sql = ", ".join(categorical_cols)

value_drift = duckdb.sql(f"""
    WITH raw AS (
        SELECT *
        FROM read_csv([{file_list_sql}], all_varchar=true, filename=true, union_by_name=true)
    ),
    unpvt AS (
        UNPIVOT raw
        ON {unpivot_cols_sql}
        INTO
            NAME column_name
            VALUE value
    )
    SELECT
        column_name,
        value,
        count(*) AS total_occurrences,
        count(DISTINCT filename) AS n_quarters_present,
        array_agg(DISTINCT regexp_extract(filename, '(\\d{{6}}_to_\\d{{6}})')) AS quarters
    FROM unpvt
    GROUP BY column_name, value
    HAVING count(DISTINCT filename) < 4
    ORDER BY column_name, n_quarters_present, value
""").df()

value_drift

,column_name,value,total_occurrences,n_quarters_present,quarters
0,AGE_GRP10,10대미만,6,1,[202308_to_202310]


##### Findings
* The set of values for each column remains consistent over time, except for the category `10대미만` (under 10 years old), indicating that the tables can be safely combined.


##### Numerical Column Distribution Drift Check
- For 11 numerical columns, compare mean/std/min/max/median across quarters
- Check whether any column's distribution shifts sharply in a specific quarter

In [ ]:
import duckdb
import pandas as pd

# 숫자형 컬럼은 값 종류가 아니라 분기별 통계(평균/표준편차/최소/최대/중앙값)를 비교해서 드리프트 확인
numeric_cols = [
    "TOTAL_USED_DAYS", "TV_SCRB", "ANALOG_SCRB", "DIGITAL_SCRB",
    "TOTAL_INTERNET_SCRB", "GIGA_INTERNET_SCRB", "TV_I_CNT",
    "CH_HH_AVG_MONTH1", "CH_25_RATIO_MONTH1", "CH_25_RATIO_MEAN_3MM",
    "KIDS_USE_PV_MONTH1",
]

unpivot_num_cols_sql = ", ".join(numeric_cols)

numeric_drift = duckdb.sql(f"""
    WITH raw AS (
        SELECT *
        FROM read_csv([{file_list_sql}], all_varchar=true, filename=true, union_by_name=true)
    ),
    unpvt_num AS (
        UNPIVOT raw
        ON {unpivot_num_cols_sql}
        INTO
            NAME column_name
            VALUE value
    )
    SELECT
        column_name,
        regexp_extract(filename, '(\\d{{6}}_to_\\d{{6}})') AS quarter,
        count(*) AS n,
        avg(TRY_CAST(value AS DOUBLE)) AS avg_val,
        stddev(TRY_CAST(value AS DOUBLE)) AS std_val,
        min(TRY_CAST(value AS DOUBLE)) AS min_val,
        max(TRY_CAST(value AS DOUBLE)) AS max_val,
        approx_quantile(TRY_CAST(value AS DOUBLE), 0.5) AS median_val
    FROM unpvt_num
    GROUP BY column_name, quarter
    ORDER BY column_name, quarter
""").df()

numeric_drift

,column_name,quarter,n,avg_val,std_val,min_val,max_val,median_val
0,ANALOG_SCRB,202302_to_202304,6257677,0.021984,0.197301,0.0,29.00,0.000000
1,ANALOG_SCRB,202305_to_202307,6250370,0.020865,0.175133,0.0,29.00,0.000000
2,ANALOG_SCRB,202308_to_202310,6237650,0.020215,0.172428,0.0,29.00,0.000000
3,ANALOG_SCRB,202311_to_202312,4147774,0.019691,0.169911,0.0,29.00,0.000000
4,CH_25_RATIO_MEAN_3MM,202302_to_202304,6257677,2.322712,5.201693,0.0,100.00,0.764483
5,CH_25_RATIO_MEAN_3MM,202305_to_202307,6250370,2.525284,5.436762,0.0,100.00,0.909747
6,CH_25_RATIO_MEAN_3MM,202308_to_202310,6237650,2.458838,5.269794,0.0,100.00,0.878681
7,CH_25_RATIO_MEAN_3MM,202311_to_202312,4147774,2.289442,5.257153,0.0,100.00,0.724056
8,CH_25_RATIO_MONTH1,202302_to_202304,6257677,2.322995,5.203052,0.0,128.87,0.764528
9,CH_25_RATIO_MONTH1,202305_to_202307,6250370,2.525546,5.437599,0.0,101.56,0.909807


##### Findings
* Max value of CH_25_RATIO_MONTH1 exceeds 100, likely due to new customers lacking a full 3 months of data.
* Max values of DIGITAL_SCRB and TV_I_CNT are large, which leads us to think that a business or bulk account (e.g., an apartment complex) may have subscribed.

##### Verify: Is CH_25_RATIO_MONTH1 > 100 Really a New-Customer Edge Case?
- Check `TOTAL_USED_DAYS` for the 10 outlier rows
- Tenure under ~90 days (3 months) directly confirms the hypothesis

In [2]:
ch_25_ratio_month1_over_100 = duckdb.sql(f"""
    SELECT count(*) AS n_over_100
    FROM read_csv([{file_list_sql}], all_varchar=true, union_by_name=true)
    WHERE TRY_CAST(CH_25_RATIO_MONTH1 AS DOUBLE) > 100
""").df()

ch_25_ratio_month1_over_100

,n_over_100
0,10


In [4]:
ch_25_ratio_month1_over_100_detail = duckdb.sql(f"""
    SELECT
        sha2_hash,
        p_mt,
        TRY_CAST(TOTAL_USED_DAYS AS DOUBLE) AS total_used_days,
        TRY_CAST(CH_25_RATIO_MONTH1 AS DOUBLE) AS ch_25_ratio_month1,
        TRY_CAST(CH_25_RATIO_MEAN_3MM AS DOUBLE) AS ch_25_ratio_mean_3mm
    FROM read_csv([{file_list_sql}], all_varchar=true, union_by_name=true)
    WHERE TRY_CAST(CH_25_RATIO_MONTH1 AS DOUBLE) > 100
    ORDER BY total_used_days
""").df()

ch_25_ratio_month1_over_100_detail

,sha2_hash,p_mt,total_used_days,ch_25_ratio_month1,ch_25_ratio_mean_3mm
0,393c97f85c39174f2f594e9c43a89c055eb8bacd3b0f0bdd6a1727e948ef13f5,202312,787.0,100.49,50.24
1,c772bc4ed8009dbee62db6441eca2227307658783d2575ffb8879d65c131a591,202302,935.0,100.81,50.41
2,be2b5c2f1f7f62c1a7768c588a0704f90234258913942d3ca0fa808bf98033a6,202305,1097.0,101.11,50.56
3,bbd6c26aa695678f9cd5466ec435e6be093838ecaeb1dcf9d766dc2b455d2c8b,202302,1378.0,128.87,64.44
4,ad5b670f6aedcde4c9ad1e482f4544a79bce4daba528ed38b9aaf74bfab616fb,202312,1601.0,121.31,60.65
5,f0235acd67253e93679239d44964709e223966628912ab8fcd2cd6825fa19953,202304,2825.0,100.55,50.27
6,ba29fac5c5dc587f95c2ba1e0cf5f6482fc7badc27a272f80d651bc2de200fae,202306,3174.0,100.33,50.17
7,9eba2a91c24bcb5afec14aec6508b40f2005a4b25a54e4e45520ea175f96a5e4,202302,3309.0,100.86,50.43
8,9eba2a91c24bcb5afec14aec6508b40f2005a4b25a54e4e45520ea175f96a5e4,202302,3309.0,100.86,50.43
9,1b456773e860d483d4e356980b52c6cc58b2fb1cc9fe2d6abd749df4b7d3e604,202306,3812.0,101.56,50.78


##### Target Variable Balance Check
- Check the class distribution of `cancel_yn` (the churn target) across the full combined dataset

In [1]:
import duckdb
import pandas as pd

files = [
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202302_to_202304 (1).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202305_to_202307 (2).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202308_to_202310 (1).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202311_to_202312 (1).csv",
]

file_list_sql = ", ".join(f"'{f}'" for f in files)

target_balance = duckdb.sql(f"""
    SELECT
        cancel_yn,
        count(*) AS n,
        count(*) * 100.0 / sum(count(*)) OVER () AS pct
    FROM read_csv([{file_list_sql}], all_varchar=true, union_by_name=true)
    GROUP BY cancel_yn
""").df()

target_balance

,cancel_yn,n,pct
0,유지,21583829,94.279408
1,해지,1309642,5.720592


##### Customer Overlap Across Quarters
- Check how many distinct quarters each customer (`sha2_hash`) appears in
- Matters for train/test splitting later: if the same customer appears in multiple quarters, splitting by row (not by customer) could leak information between train and test

In [2]:
import duckdb
import pandas as pd

files = [
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202302_to_202304 (1).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202305_to_202307 (2).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202308_to_202310 (1).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202311_to_202312 (1).csv",
]

file_list_sql = ", ".join(f"'{f}'" for f in files)

customer_overlap = duckdb.sql(f"""
    WITH raw AS (
        SELECT sha2_hash, filename
        FROM read_csv([{file_list_sql}], all_varchar=true, filename=true, union_by_name=true)
    ),
    customer_quarters AS (
        SELECT sha2_hash, count(DISTINCT filename) AS n_quarters_present
        FROM raw
        GROUP BY sha2_hash
    )
    SELECT n_quarters_present, count(*) AS n_customers
    FROM customer_quarters
    GROUP BY n_quarters_present
    ORDER BY n_quarters_present
""").df()

customer_overlap

,n_quarters_present,n_customers
0,1,66257
1,2,80430
2,3,77768
3,4,1950872


##### Why exclude these 33 rows from STB_RES_1M_YN 
- Contract fields (e.g., `cancel_yn`) are populated normally, but customer profile fields (age, product tier, etc.) remain empty throughout — suggesting these are internal test/special accounts rather than real customers, so they are excluded.

In [1]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

files = [
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202302_to_202304 (1).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202305_to_202307 (2).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202308_to_202310 (1).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202311_to_202312 (1).csv",
]
file_list_sql = ", ".join(f"'{f}'" for f in files)

corrupted_rows = duckdb.sql(f"""
    SELECT *
    FROM read_csv([{file_list_sql}], all_varchar=true, filename=true, union_by_name=true)
    WHERE STB_RES_1M_YN = '""'
    ORDER BY sha2_hash, p_mt
""").df()

corrupted_rows

,sha2_hash,SVC_USE_DAYS_GRP,MEDIA_NM_GRP,PROD_NM_GRP,PROD_OLD_YN,PROD_ONE_PLUS_YN,AGMT_KIND_NM,STB_RES_1M_YN,SVOD_SCRB_CNT_GRP,PAID_CHNL_CNT_GRP,SCRB_PATH_NM_GRP,INHOME_RATE,AGMT_END_SEG,AGMT_END_YMD,TOTAL_USED_DAYS,TV_SCRB,ANALOG_SCRB,DIGITAL_SCRB,TOTAL_INTERNET_SCRB,GIGA_INTERNET_SCRB,BUNDLE_YN,DIGITAL_GIGA_YN,DIGITAL_ALOG_YN,TV_I_CNT,CH_LAST_DAYS_BF_GRP,VOC_TOTAL_MONTH1_YN,VOC_STOP_CANCEL_MONTH1_YN,AGE_GRP10,EMAIL_RECV_CLS_NM,SMS_SEND_CLS_NM,CH_HH_AVG_MONTH1,CH_25_RATIO_MONTH1,CH_25_RATIO_MEAN_3MM,CH_FAV_RNK1,KIDS_USE_PV_MONTH1,NFX_USE_YN,YTB_USE_YN,p_mt,cancel_yn,filename
0,07986f40d95a5b0ee967be782e212dc73cd223a231c0a43928674734562761e8,36개월 이상,기타,기타,N,N,정보없음,"""""",기타,기타,정보없음,"""""",기타,정보없음,2955,"""""","""""","""""","""""","""""",N,N,N,"""""",3개월내없음,N,N,"""""",정보없음,정보없음,0.0,0.0,0.0,기타,0.0,N,N,202302,유지,LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202302_to_202304 (1).csv
1,07986f40d95a5b0ee967be782e212dc73cd223a231c0a43928674734562761e8,36개월 이상,기타,기타,N,N,정보없음,"""""",기타,기타,정보없음,"""""",기타,정보없음,2986,"""""","""""","""""","""""","""""",N,N,N,"""""",3개월내없음,N,N,"""""",정보없음,정보없음,0.0,0.0,0.0,기타,0.0,N,N,202303,유지,LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202302_to_202304 (1).csv
2,07986f40d95a5b0ee967be782e212dc73cd223a231c0a43928674734562761e8,36개월 이상,기타,기타,N,N,정보없음,"""""",기타,기타,정보없음,"""""",기타,정보없음,3016,"""""","""""","""""","""""","""""",N,N,N,"""""",3개월내없음,N,N,"""""",정보없음,정보없음,0.0,0.0,0.0,기타,0.0,N,N,202304,유지,LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202302_to_202304 (1).csv
3,07986f40d95a5b0ee967be782e212dc73cd223a231c0a43928674734562761e8,36개월 이상,기타,기타,N,N,정보없음,"""""",기타,기타,정보없음,"""""",기타,정보없음,3047,"""""","""""","""""","""""","""""",N,N,N,"""""",3개월내없음,N,N,"""""",정보없음,정보없음,0.0,0.0,0.0,기타,0.0,N,N,202305,유지,LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202305_to_202307 (2).csv
4,07986f40d95a5b0ee967be782e212dc73cd223a231c0a43928674734562761e8,36개월 이상,기타,기타,N,N,정보없음,"""""",기타,기타,정보없음,"""""",기타,정보없음,3077,"""""","""""","""""","""""","""""",N,N,N,"""""",3개월내없음,N,N,"""""",정보없음,정보없음,0.0,0.0,0.0,기타,0.0,N,N,202306,유지,LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202305_to_202307 (2).csv
5,07986f40d95a5b0ee967be782e212dc73cd223a231c0a43928674734562761e8,36개월 이상,기타,기타,N,N,정보없음,"""""",기타,기타,정보없음,"""""",기타,정보없음,3108,"""""","""""","""""","""""","""""",N,N,N,"""""",3개월내없음,N,N,"""""",정보없음,정보없음,0.0,0.0,0.0,기타,0.0,N,N,202307,유지,LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202305_to_202307 (2).csv
6,07986f40d95a5b0ee967be782e212dc73cd223a231c0a43928674734562761e8,36개월 이상,기타,기타,N,N,정보없음,"""""",기타,기타,정보없음,"""""",기타,정보없음,3139,"""""","""""","""""","""""","""""",N,N,N,"""""",3개월내없음,N,N,"""""",정보없음,정보없음,0.0,0.0,0.0,기타,0.0,N,N,202308,유지,LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202308_to_202310 (1).csv
7,07986f40d95a5b0ee967be782e212dc73cd223a231c0a43928674734562761e8,36개월 이상,기타,기타,N,N,정보없음,"""""",기타,기타,정보없음,"""""",기타,정보없음,3169,"""""","""""","""""","""""","""""",N,N,N,"""""",3개월내없음,N,N,"""""",정보없음,정보없음,0.0,0.0,0.0,기타,0.0,N,N,202309,유지,LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202308_to_202310 (1).csv
8,07986f40d95a5b0ee967be782e212dc73cd223a231c0a43928674734562761e8,36개월 이상,기타,기타,N,N,정보없음,"""""",기타,기타,정보없음,"""""",기타,정보없음,3200,"""""","""""","""""","""""","""""",N,N,N,"""""",3개월내없음,N,N,"""""",정보없음,정보없음,0.0,0.0,0.0,기타,0.0,N,N,202310,유지,LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202308_to_202310 (1).csv
9,07986f40d95a5b0ee967be782e212dc73cd223a231c0a43928674734562761e8,36개월 이상,기타,기타,N,N,정보없음,"""""",기타,기타,정보없음,"""""",기타,정보없음,3230,"""""","""""","""""","""""","""""",N,N,N,"""""",3개월내없음,N,N,"""""",정보없음,정보없음,0.0,0.0,0.0,기타,0.0,N,N,202311,유지,LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202311_to_202312 (1).csv


##### Confirm: Is SVOD_SCRB_CNT_GRP = '기타' the Same 33 Rows?
- Both `STB_RES_1M_YN = '""'` and `SVOD_SCRB_CNT_GRP = '기타'` show exactly 33 rows — check whether they fully overlap (same 33 rows) or it's just a coincidental matching count

In [ ]:
svod_overlap_check = duckdb.sql(f"""
    SELECT
        count(*) FILTER (WHERE STB_RES_1M_YN = '""' AND SVOD_SCRB_CNT_GRP = '기타') AS both,
        count(*) FILTER (WHERE STB_RES_1M_YN = '""' AND SVOD_SCRB_CNT_GRP != '기타') AS stb_only,
        count(*) FILTER (WHERE STB_RES_1M_YN != '""' AND SVOD_SCRB_CNT_GRP = '기타') AS svod_only
    FROM read_csv([{file_list_sql}], all_varchar=true, union_by_name=true)
""").df()

svod_overlap_check

##### Duplicate Row Check
- Group by `sha2_hash` + `p_mt` and count how many rows share the same key
- Rows in a group of size > 1 are "involved" in duplication, but only `n_dup - 1` per group are actual excess rows to drop

In [3]:
dup_summary = duckdb.sql(f"""
    WITH raw AS (
        SELECT * EXCLUDE (filename) FROM read_csv([{file_list_sql}], all_varchar=true, filename=true, union_by_name=true)
    ),
    key_counts AS (
        SELECT sha2_hash, p_mt, count(*) AS n_rows_per_key
        FROM raw
        GROUP BY sha2_hash, p_mt
    )
    SELECT
        sum(n_rows_per_key) AS total_rows,
        sum(CASE WHEN n_rows_per_key > 1 THEN n_rows_per_key ELSE 0 END) AS conflicted_rows,
        sum(n_rows_per_key - 1) AS excess_rows_to_remove,
        round(sum(CASE WHEN n_rows_per_key > 1 THEN n_rows_per_key ELSE 0 END) * 100.0 / sum(n_rows_per_key), 2) AS pct_conflicted_rows,
        round(sum(n_rows_per_key - 1) * 100.0 / sum(n_rows_per_key), 2) AS pct_excess_rows
    FROM key_counts
""").df()

dup_summary

,total_rows,conflicted_rows,excess_rows_to_remove,pct_conflicted_rows,pct_excess_rows
0,22893471.0,839874.0,419937.0,3.67,1.83


In [3]:
exact_dup_summary = duckdb.sql(f"""
    WITH raw AS (
        SELECT * EXCLUDE (filename) FROM read_csv([{file_list_sql}], all_varchar=true, filename=true, union_by_name=true)
    ),
    grouped AS (
        SELECT *, count(*) AS n_exact_dup
        FROM raw
        GROUP BY ALL
    )
    SELECT
        count(*) FILTER (WHERE n_exact_dup > 1) AS n_dup_groups,
        sum(n_exact_dup) FILTER (WHERE n_exact_dup > 1) AS rows_in_exact_dup_groups,
        sum(n_exact_dup - 1) FILTER (WHERE n_exact_dup > 1) AS excess_exact_dup_rows
    FROM grouped
""").df()

exact_dup_summary

,n_dup_groups,rows_in_exact_dup_groups,excess_exact_dup_rows
0,0,NaN,NaN


##### Which Columns Differ Within sha2_hash + p_mt Duplicate Groups?
- For each duplicate-key group (same `sha2_hash` + `p_mt`, count > 1), check which of the other columns have more than 1 distinct value
- If a column shows up here, the duplicate rows are NOT exact copies — they conflict on that column

In [6]:
other_cols = [
    "SVC_USE_DAYS_GRP", "MEDIA_NM_GRP", "PROD_NM_GRP", "PROD_OLD_YN", "PROD_ONE_PLUS_YN",
    "AGMT_KIND_NM", "STB_RES_1M_YN", "SVOD_SCRB_CNT_GRP", "PAID_CHNL_CNT_GRP", "SCRB_PATH_NM_GRP",
    "INHOME_RATE", "AGMT_END_SEG", "AGMT_END_YMD", "TOTAL_USED_DAYS", "TV_SCRB", "ANALOG_SCRB",
    "DIGITAL_SCRB", "TOTAL_INTERNET_SCRB", "GIGA_INTERNET_SCRB", "BUNDLE_YN", "DIGITAL_GIGA_YN",
    "DIGITAL_ALOG_YN", "TV_I_CNT", "CH_LAST_DAYS_BF_GRP", "VOC_TOTAL_MONTH1_YN",
    "VOC_STOP_CANCEL_MONTH1_YN", "AGE_GRP10", "EMAIL_RECV_CLS_NM", "SMS_SEND_CLS_NM",
    "CH_HH_AVG_MONTH1", "CH_25_RATIO_MONTH1", "CH_25_RATIO_MEAN_3MM", "CH_FAV_RNK1",
    "KIDS_USE_PV_MONTH1", "NFX_USE_YN", "YTB_USE_YN", "cancel_yn",
]
other_cols_sql = ", ".join(other_cols)

dup_col_diff = duckdb.sql(f"""
    WITH raw AS (
        SELECT * EXCLUDE (filename) FROM read_csv([{file_list_sql}], all_varchar=true, filename=true, union_by_name=true)
    ),
    dup_keys AS (
        SELECT sha2_hash, p_mt
        FROM raw
        GROUP BY sha2_hash, p_mt
        HAVING count(*) > 1
    ),
    dup_rows AS (
        SELECT raw.*
        FROM raw
        JOIN dup_keys USING (sha2_hash, p_mt)
    ),
    unpvt AS (
        UNPIVOT dup_rows
        ON {other_cols_sql}
        INTO
            NAME column_name
            VALUE value
    ),
    group_distinct AS (
        SELECT sha2_hash, p_mt, column_name, count(DISTINCT value) AS n_distinct
        FROM unpvt
        GROUP BY sha2_hash, p_mt, column_name
    )
    SELECT column_name, count(*) AS n_groups_with_diff
    FROM group_distinct
    WHERE n_distinct > 1
    GROUP BY column_name
    ORDER BY n_groups_with_diff DESC
""").df()

dup_col_diff

,column_name,n_groups_with_diff
0,cancel_yn,419937


##### Show One Real Conflicting Customer as an Example
- Pick a single `sha2_hash + p_mt` conflict group and print both of its rows in full, to see the actual conflict with real values instead of just aggregate counts

In [7]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

one_conflict_example = duckdb.sql(f"""
    WITH raw AS (
        SELECT * EXCLUDE (filename) FROM read_csv([{file_list_sql}], all_varchar=true, filename=true, union_by_name=true)
    ),
    one_dup_key AS (
        SELECT sha2_hash, p_mt
        FROM raw
        GROUP BY sha2_hash, p_mt
        HAVING count(*) > 1
        LIMIT 1
    )
    SELECT raw.*
    FROM raw
    JOIN one_dup_key USING (sha2_hash, p_mt)
""").df()

one_conflict_example

,sha2_hash,SVC_USE_DAYS_GRP,MEDIA_NM_GRP,PROD_NM_GRP,PROD_OLD_YN,PROD_ONE_PLUS_YN,AGMT_KIND_NM,STB_RES_1M_YN,SVOD_SCRB_CNT_GRP,PAID_CHNL_CNT_GRP,SCRB_PATH_NM_GRP,INHOME_RATE,AGMT_END_SEG,AGMT_END_YMD,TOTAL_USED_DAYS,TV_SCRB,ANALOG_SCRB,DIGITAL_SCRB,TOTAL_INTERNET_SCRB,GIGA_INTERNET_SCRB,BUNDLE_YN,DIGITAL_GIGA_YN,DIGITAL_ALOG_YN,TV_I_CNT,CH_LAST_DAYS_BF_GRP,VOC_TOTAL_MONTH1_YN,VOC_STOP_CANCEL_MONTH1_YN,AGE_GRP10,EMAIL_RECV_CLS_NM,SMS_SEND_CLS_NM,CH_HH_AVG_MONTH1,CH_25_RATIO_MONTH1,CH_25_RATIO_MEAN_3MM,CH_FAV_RNK1,KIDS_USE_PV_MONTH1,NFX_USE_YN,YTB_USE_YN,p_mt,cancel_yn
0,b2927b717dc6460ff9471bbfbc65dc866546c5eca59842e78423716062a60eb2,36개월 이상,HD,프리미엄,N,N,약정승계,N,0건,2건,O/B,20.0,약정만료후 12개월이상,20190824,2409,2,0,2,1,1,Y,Y,N,3.0,일주일내,N,N,50대,전체거부,전체거부,4.38,2.58,2.58,TV조선,0.0,N,N,202303,해지
1,b2927b717dc6460ff9471bbfbc65dc866546c5eca59842e78423716062a60eb2,36개월 이상,HD,프리미엄,N,N,약정승계,N,0건,2건,O/B,20.0,약정만료후 12개월이상,20190824,2409,2,0,2,1,1,Y,Y,N,3.0,일주일내,N,N,50대,전체거부,전체거부,4.38,2.58,2.58,TV조선,0.0,N,N,202303,유지


##### Do Conflicting Customers Have a Later Record?
- For each `sha2_hash + p_mt` conflict group, check whether that customer has any row with a later `p_mt` anywhere in the dataset
- No later record → the customer likely left for good after this month (leans `해지`)
- Has a later record → the customer kept appearing afterward (leans `유지`, i.e. cancellation was reversed)

In [7]:
conflict_later_record_check = duckdb.sql(f"""
    WITH raw AS (
        SELECT sha2_hash, p_mt
        FROM read_csv([{file_list_sql}], all_varchar=true, filename=true, union_by_name=true)
    ),
    dup_keys AS (
        SELECT sha2_hash, p_mt
        FROM raw
        GROUP BY sha2_hash, p_mt
        HAVING count(*) > 1
    ),
    customer_max_pmt AS (
        SELECT sha2_hash, max(TRY_CAST(p_mt AS INTEGER)) AS max_pmt
        FROM raw
        GROUP BY sha2_hash
    )
    SELECT
        count(*) AS n_conflict_groups,
        count(*) FILTER (WHERE TRY_CAST(dk.p_mt AS INTEGER) = cm.max_pmt) AS n_no_later_record,
        count(*) FILTER (WHERE TRY_CAST(dk.p_mt AS INTEGER) < cm.max_pmt) AS n_has_later_record
    FROM dup_keys dk
    JOIN customer_max_pmt cm USING (sha2_hash)
""").df()

conflict_later_record_check

,n_conflict_groups,n_no_later_record,n_has_later_record
0,419937,40063,379874


##### What Does the Next Month's cancel_yn Actually Say?
- For the 379,874 conflict groups that have a later record, find that customer's very next `p_mt` and check its `cancel_yn`
- If the next month says `해지`, the conflicting month was likely a true cancellation (keep `해지`)
- If the next month says `유지`, the cancellation was likely reversed (keep `유지`)

In [9]:
next_month_status_check = duckdb.sql(f"""
    WITH raw_int AS (
        SELECT sha2_hash, TRY_CAST(p_mt AS INTEGER) AS p_mt_int, cancel_yn
        FROM read_csv([{file_list_sql}], all_varchar=true, filename=true, union_by_name=true)
    ),
    dup_keys AS (
        SELECT sha2_hash, p_mt_int
        FROM raw_int
        GROUP BY sha2_hash, p_mt_int
        HAVING count(*) > 1
    ),
    next_pmt AS (
        SELECT dk.sha2_hash, dk.p_mt_int AS conflict_pmt, min(r.p_mt_int) AS next_pmt
        FROM dup_keys dk
        JOIN raw_int r
          ON r.sha2_hash = dk.sha2_hash AND r.p_mt_int > dk.p_mt_int
        GROUP BY dk.sha2_hash, dk.p_mt_int
    ),
    next_status AS (
        SELECT
            np.sha2_hash,
            np.conflict_pmt,
            np.next_pmt,
            count(DISTINCT r.cancel_yn) AS n_distinct_next_status,
            array_agg(DISTINCT r.cancel_yn) AS next_status_values
        FROM next_pmt np
        JOIN raw_int r
          ON r.sha2_hash = np.sha2_hash AND r.p_mt_int = np.next_pmt
        GROUP BY np.sha2_hash, np.conflict_pmt, np.next_pmt
    )
    SELECT
        CASE
            WHEN n_distinct_next_status > 1 THEN '다음달도_충돌'
            ELSE next_status_values[1]
        END AS next_month_status,
        count(*) AS n,
        round(count(*) * 100.0 / sum(count(*)) OVER (), 2) AS pct
    FROM next_status
    GROUP BY 1
    ORDER BY n DESC
""").df()

next_month_status_check

,next_month_status,n,pct
0,다음달도_충돌,379874,100.0


##### How Many Customers Are Affected, and Does the Duplication Persist?
- Count distinct `sha2_hash` that ever have a duplicated month
- For each affected customer, check every month from their first conflict onward: is it *always* duplicated (n_rows > 1) or does it sometimes drop back to a single row (a "gap")?

In [10]:
affected_customer_persistence_check = duckdb.sql(f"""
    WITH raw_int AS (
        SELECT sha2_hash, TRY_CAST(p_mt AS INTEGER) AS p_mt_int
        FROM read_csv([{file_list_sql}], all_varchar=true, filename=true, union_by_name=true)
    ),
    per_customer_month AS (
        SELECT sha2_hash, p_mt_int, count(*) AS n_rows
        FROM raw_int
        GROUP BY sha2_hash, p_mt_int
    ),
    customer_first_conflict AS (
        SELECT sha2_hash, min(p_mt_int) AS first_conflict_pmt
        FROM per_customer_month
        WHERE n_rows > 1
        GROUP BY sha2_hash
    ),
    months_after_first_conflict AS (
        SELECT pcm.sha2_hash, min(pcm.n_rows) AS min_n_rows_after
        FROM per_customer_month pcm
        JOIN customer_first_conflict cfc
          ON pcm.sha2_hash = cfc.sha2_hash AND pcm.p_mt_int >= cfc.first_conflict_pmt
        GROUP BY pcm.sha2_hash
    )
    SELECT
        (SELECT count(DISTINCT sha2_hash) FROM raw_int) AS n_total_customers,
        count(*) AS n_affected_customers,
        round(count(*) * 100.0 / (SELECT count(DISTINCT sha2_hash) FROM raw_int), 2) AS pct_affected,
        count(*) FILTER (WHERE min_n_rows_after > 1) AS n_always_dup_after_first,
        count(*) FILTER (WHERE min_n_rows_after = 1) AS n_has_gap_after_first
    FROM months_after_first_conflict
""").df()

affected_customer_persistence_check

,n_total_customers,n_affected_customers,pct_affected,n_always_dup_after_first,n_has_gap_after_first
0,2175327,40063,1.84,40063,0


##### Are the Affected 40,063 Customers Business/Bulk Accounts?
- `TV_I_CNT` (installed TV unit count) is the most direct physical signal for "many rooms/devices under one account" — check its avg/max between the affected group and everyone else

In [12]:
tv_i_cnt_check = duckdb.sql(f"""
    WITH raw AS (
        SELECT
            sha2_hash,
            p_mt,
            TRY_CAST(TV_I_CNT AS DOUBLE) AS tv_i_cnt
        FROM read_csv([{file_list_sql}], all_varchar=true, filename=true, union_by_name=true)
    ),
    dup_pmt_keys AS (
        SELECT sha2_hash, p_mt
        FROM raw
        GROUP BY sha2_hash, p_mt
        HAVING count(*) > 1
    ),
    affected_customers AS (
        SELECT DISTINCT sha2_hash FROM dup_pmt_keys
    ),
    labeled AS (
        SELECT
            raw.*,
            CASE WHEN ac.sha2_hash IS NOT NULL THEN 'affected' ELSE 'normal' END AS group_label
        FROM raw
        LEFT JOIN affected_customers ac USING (sha2_hash)
    )
    SELECT
        group_label,
        count(*) AS n_rows,
        count(DISTINCT sha2_hash) AS n_customers,
        round(avg(tv_i_cnt), 2) AS avg_tv_i_cnt,
        max(tv_i_cnt) AS max_tv_i_cnt
    FROM labeled
    GROUP BY group_label
""").df()

tv_i_cnt_check

,group_label,n_rows,n_customers,avg_tv_i_cnt,max_tv_i_cnt
0,affected,839874,40063,2.61,91.0
1,normal,22053597,2135264,2.27,275.0


##### Fixing a Sample-Size Bias: Compare Rates, Not Raw Max
- `normal` has ~53x more customers than `affected` (2,135,264 vs 40,063), so it's far more likely to happen to contain a rare extreme outlier just by chance — the max comparison above isn't a fair test
- Instead, compare the **rate** of "large" accounts (TV_I_CNT ≥ 10) within each group — a proportion isn't biased by group size the way a raw max is

In [13]:
bulk_rate_check = duckdb.sql(f"""
    WITH raw AS (
        SELECT
            sha2_hash,
            p_mt,
            TRY_CAST(TV_I_CNT AS DOUBLE) AS tv_i_cnt
        FROM read_csv([{file_list_sql}], all_varchar=true, filename=true, union_by_name=true)
    ),
    dup_pmt_keys AS (
        SELECT sha2_hash, p_mt
        FROM raw
        GROUP BY sha2_hash, p_mt
        HAVING count(*) > 1
    ),
    affected_customers AS (
        SELECT DISTINCT sha2_hash FROM dup_pmt_keys
    ),
    labeled AS (
        SELECT
            raw.*,
            CASE WHEN ac.sha2_hash IS NOT NULL THEN 'affected' ELSE 'normal' END AS group_label
        FROM raw
        LEFT JOIN affected_customers ac USING (sha2_hash)
    )
    SELECT
        group_label,
        count(*) AS n_rows,
        count(*) FILTER (WHERE tv_i_cnt >= 10) AS n_rows_bulk,
        round(count(*) FILTER (WHERE tv_i_cnt >= 10) * 100.0 / count(*), 4) AS pct_rows_bulk
    FROM labeled
    GROUP BY group_label
""").df()

bulk_rate_check

,group_label,n_rows,n_rows_bulk,pct_rows_bulk
0,affected,839874,13306,1.5843
1,normal,22053597,170999,0.7754


##### Findings
* Bulk/business accounts (`TV_I_CNT` ≥ 10) are ~2x more common in the affected group than normal (1.58% vs 0.78% of rows) — a real but partial contributing factor.
* Still, 98%+ of affected customers are ordinary-sized accounts, so bulk accounts don't explain most of the pattern. Root cause remains largely undetermined — decision to exclude these 40,063 customers stands.